# Azure Document Intelligence - invoice smoke test

Set `AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT` and `AZURE_DOCUMENT_INTELLIGENCE_KEY`, or enter them when prompted. This uses the `prebuilt-invoice` model against a public sample invoice.

In [1]:
%pip install -q azure-ai-documentintelligence==1.0.2 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import os

endpoint = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT") or input("Endpoint: ").strip()
key = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_KEY") or getpass.getpass("Key: ")

assert endpoint and key, "Set endpoint and key before continuing"

In [3]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential

client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

invoice_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"
poller = client.begin_analyze_document(
    "prebuilt-invoice",
    AnalyzeDocumentRequest(url_source=invoice_url),
)
result = poller.result()

len(result.documents), result.model_id

(1, 'prebuilt-invoice')

In [4]:
from IPython.display import JSON, display

doc = result.documents[0] if result.documents else None

def compact(field):
    if not field:
        return None
    return {
        "content": field.get("content"),
        "confidence": field.get("confidence"),
        "type": field.get("type"),
    }

summary_fields = [
    "VendorName",
    "CustomerName",
    "InvoiceId",
    "InvoiceDate",
    "DueDate",
    "PurchaseOrder",
    "SubTotal",
    "TotalTax",
    "InvoiceTotal",
    "AmountDue",
]

summary = {}
items = []

if doc:
    summary = {
        name: compact(doc.fields.get(name))
        for name in summary_fields
        if doc.fields.get(name)
    }

    items_field = doc.fields.get("Items")
    for item in (items_field or {}).get("valueArray", []):
        value_object = item.get("valueObject", {})
        items.append({
            name: compact(value_object.get(name))
            for name in ["Description", "Quantity", "UnitPrice", "Amount"]
            if value_object.get(name)
        })

display(JSON({"summary": summary, "items": items}, expanded=True))

<IPython.core.display.JSON object>

In [ ]:
# Optional: analyze a local invoice instead of the sample URL.
# import base64
# local_path = r"C:\path\to\invoice.pdf"
# with open(local_path, "rb") as f:
#     body = AnalyzeDocumentRequest(bytes_source=base64.b64encode(f.read()))
# result = client.begin_analyze_document("prebuilt-invoice", body).result()